In [1]:
import os
os.environ['JAVA_HOME'] = "/usr/lib/jvm/java-21-openjdk-amd64"

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import RegexTokenizer, HashingTF, IDF
from pyspark.sql.functions import col, udf, lit, regexp_extract, coalesce, when
from pyspark.sql.types import FloatType, IntegerType, StringType
import numpy as np
spark = SparkSession.builder.master('local[*]').getOrCreate()

spark.conf.set('spark.sql.repl.eagerEval.enabled', True)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 15:30:50 WARN Utils: Your hostname, comp, resolves to a loopback address: 127.0.1.1; using 192.168.0.108 instead (on interface wlo1)
26/06/13 15:30:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 15:30:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# load dataset 
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .load("dataset/glints_dataset.csv")

df.printSchema()
df.show(5)

root
 |-- job_title: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- education_req: string (nullable = true)
 |-- salary_range: string (nullable = true)
 |-- job_requirements: string (nullable = true)
 |-- job_responsibilities: string (nullable = true)
 |-- posted_date: timestamp (nullable = true)
 |-- source_platform: string (nullable = true)
 |-- category_scraped: string (nullable = true)

+--------------------+--------------------+----------------+---------+----------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+
|           job_title|        company_name|        location| job_type|experience_level|  education_req|        salary_range|    job_requirements|job_responsibilities|         posted_date|source_platform|    category_scraped|
+

In [4]:
df.printSchema()

root
 |-- job_title: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- education_req: string (nullable = true)
 |-- salary_range: string (nullable = true)
 |-- job_requirements: string (nullable = true)
 |-- job_responsibilities: string (nullable = true)
 |-- posted_date: timestamp (nullable = true)
 |-- source_platform: string (nullable = true)
 |-- category_scraped: string (nullable = true)



In [5]:
from pyspark.sql.functions import col, sum

# Get the count of nulls in each column
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+---------+------------+--------+--------+----------------+-------------+------------+----------------+--------------------+-----------+---------------+----------------+
|job_title|company_name|location|job_type|experience_level|education_req|salary_range|job_requirements|job_responsibilities|posted_date|source_platform|category_scraped|
+---------+------------+--------+--------+----------------+-------------+------------+----------------+--------------------+-----------+---------------+----------------+
|        0|           0|       0|       0|               0|            0|           0|               0|                   0|          0|              0|               0|
+---------+------------+--------+--------+----------------+-------------+------------+----------------+--------------------+-----------+---------------+----------------+



In [6]:
df.select('job_type').distinct().show()

+-------------+
|     job_type|
+-------------+
|    PART_TIME|
|    FULL_TIME|
|   INTERNSHIP|
|     CONTRACT|
|PROJECT_BASED|
+-------------+



In [7]:
# shows unique value in a column in dataset
df.select("education_req").distinct().show(truncate=False)

+----------------------+
|education_req         |
+----------------------+
|SECONDARY_SCHOOL      |
|PRIMARY_SCHOOL        |
|MASTER_DEGREE         |
|DIPLOMA               |
|PROFESSIONAL_EDUCATION|
|DOCTORATE             |
|BACHELOR_DEGREE       |
|HIGH_SCHOOL           |
+----------------------+



In [8]:
df.select("location").distinct().show(truncate=False)

+----------------+
|location        |
+----------------+
|Tebet           |
|Purwodadi       |
|Menteng         |
|Batu Raja Timur |
|Rajeg           |
|Cicalengka      |
|Ulujadi         |
|Pangkalan Baru  |
|Panarukan       |
|Balige          |
|Mundu           |
|Pasawahan       |
|Tangerang       |
|Kemantren Kraton|
|Padang Barat    |
|Mejayan         |
|Bukitintan      |
|Bojong          |
|Tempe           |
|Gatak           |
+----------------+
only showing top 20 rows


In [9]:
# register dataframe as a temporary sql table
df.createOrReplaceGlobalTempView('my_table')

# Preprocessing

In [10]:
# drop duplicate data and data where scrapper failed to fetch (starts with "Error:", usually caused by network issue)
def clean_df(df):
    df_clean = df.dropDuplicates().filter(~col("job_responsibilities").startswith("Error:"))

    # drop all data with null values
    df_clean = df_clean.dropna()

    return df_clean

df_clean = clean_df(df)

print("Amount of rows before cleaning:", df.count())
print("Amount of rows after cleaning:", df_clean.count())

Amount of rows before cleaning: 75748


Amount of rows after cleaning: 70486


In [11]:
# create new columns: min_experience_level, max_experience_level, min_salary, dan max_salary
def create_cols(df):
    raw_min_exp = regexp_extract(col("experience_level"), r"(\d+)", 1)
    raw_max_exp = regexp_extract(col("experience_level"), r"-(\d+)", 1)
    
    raw_min_sal = regexp_extract(col("salary_range"), r"(\d+)", 1)
    raw_max_sal = regexp_extract(col("salary_range"), r"-\s*(\d+)", 1)

    df_new = df.withColumn(
        "min_experience_level",
        when(raw_min_exp == "", None).otherwise(raw_min_exp).cast("int")
    ).withColumn(
        "max_experience_level",
        coalesce(
            when(raw_max_exp == "", None).otherwise(raw_max_exp).cast("int"),
            col("min_experience_level")
        )
    ).withColumn(
        "min_salary",
        when(raw_min_sal == "", None).otherwise(raw_min_sal).cast("long")
    ).withColumn(
        "max_salary",
        coalesce(
            when(raw_max_sal == "", None).otherwise(raw_max_sal).cast("long"),
            col("min_salary")
        )
    ).drop("salary_range", "experience_level")

    return df_new

df_new = create_cols(df_clean)

In [12]:
def encode_edu_req(education):
    education_map = {
        "PRIMARY_SCHOOL": 1,
        "SECONDARY_SCHOOL": 2,
        "HIGH_SCHOOL": 3,
        "DIPLOMA": 4,
        "BACHELOR_DEGREE": 5,
        "PROFESSIONAL_EDUCATION": 6,
        "MASTER_DEGREE": 7,
        "DOCTORATE": 8
    }

    if education is None:
        return 0
    return education_map.get(education.upper(), 0)

def decode_edu_req(education_rank):
    inverse_education_map = {
        1: "PRIMARY_SCHOOL",
        2: "SECONDARY_SCHOOL",
        3: "HIGH_SCHOOL",
        4: "DIPLOMA",
        5: "BACHELOR_DEGREE",
        6: "PROFESSIONAL_EDUCATION",
        7: "MASTER_DEGREE",
        8: "DOCTORATE"
    }
    
    if education_rank is None:
        return "UNKNOWN"
        
    return inverse_education_map.get(int(education_rank), "UNKNOWN")

# register function as an User Definedd Function (UDF) with Integer as the output type
encode_edu_udf = udf(encode_edu_req, IntegerType())

df_encoded = df_new.withColumn("education_req", encode_edu_udf(col("education_req")))

df_encoded.show()

+--------------------+--------------------+---------------+---------+-------------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+----------+----------+
|           job_title|        company_name|       location| job_type|education_req|    job_requirements|job_responsibilities|         posted_date|source_platform|    category_scraped|min_experience_level|max_experience_level|min_salary|max_salary|
+--------------------+--------------------+---------------+---------+-------------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+----------+----------+
|HELPER DRIVER / W...|PT Shield On Serv...|         Gambir|FULL_TIME|            3|Stock Opname, Log...|Deskripsi pekerja...|2026-03-10 11:28:...|         Glints|Supply Chain, Log...|                   3|                   5|   4000000|   4000000|
|Helper 

In [13]:
# USER INPUT (can be configured)
user_skills = "Microsoft Excel, Data Entry, Administration, Problem Solving"
user_years_of_experience = 1
user_highest_education = "HIGH_SCHOOL"
user_job_type = None
user_salary_target = 5000000
user_job_category = None

def df_user(df, job_type=None, highest_education=None, years_of_experience=None, salary_target=None, job_category=None):
    df_filtered = df
    if job_type is not None:
        df_filtered = df_filtered.filter(col("job_type") == job_type)
    
    if highest_education is not None:
        df_filtered = df_filtered.filter(col("education_req") <= highest_education)
    
    if years_of_experience is not None:
        df_filtered = df_filtered.filter(
            (col("min_experience_level") <= years_of_experience) &
            (col("max_experience_level") >= years_of_experience)
        )

    if salary_target is not None:
        df_filtered = df_filtered = df_filtered.filter(
            (col("min_salary") <= salary_target) & 
            (col("max_salary") >= salary_target)
        )

    if job_category is not None:
        df_filtered = df_filtered = df_filtered.filter(col("job_category") == job_category)
        
    return df_filtered

df_filtered = df_user(
    df_encoded,
    job_type=user_job_type,
    highest_education=encode_edu_req(user_highest_education),
    years_of_experience=user_years_of_experience,
    salary_target=user_salary_target,
    job_category=user_job_category
)

In [14]:
# tokenized skill requirement of an organization/company
tokenizer = RegexTokenizer(inputCol="job_requirements", outputCol="skills_tokens", pattern=",\s*")

df_tokenized = tokenizer.transform(df_filtered)

In [15]:
# vectorized tokens
hashingTF = HashingTF(inputCol="skills_tokens", outputCol="rawFeatures", numFeatures=1000)
featurizedData = hashingTF.transform(df_tokenized)

idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)

rescaledData = idfModel.transform(featurizedData)

In [16]:
# tokenized and vectorized user skills
user_df = spark.createDataFrame([(user_skills,)], ["job_requirements"])

user_tokenized = tokenizer.transform(user_df)
user_featurized = hashingTF.transform(user_tokenized)
user_rescaled = idfModel.transform(user_featurized)

user_vector = user_rescaled.select("features").head()[0]

In [17]:
# define function to calculate similarity with cosine similarity formula
def cosine_similarity(v1):
    dot_product = float(v1.dot(user_vector))
    norm_v1 = float(v1.norm(2))
    norm_user = float(user_vector.norm(2))
    
    if norm_v1 == 0 or norm_user == 0:
        return 0.0
        
    return dot_product / (norm_v1 * norm_user)

cosine_similarity_udf = udf(cosine_similarity, FloatType())

In [18]:
# comparing and calculate the similarity of user skills and required skills
df_with_similarity = rescaledData.withColumn("similarity_score", cosine_similarity_udf(col("features")))

df_sorted = df_with_similarity.orderBy(col("similarity_score").desc())

# decode education level from integer to a human-readable string
decode_edu_udf = udf(decode_edu_req, StringType())
df_sorted = df_sorted.withColumn("education_req", decode_edu_udf(col("education_req")))

# shows top 10 recommendations (can be configured)
TOP_RECOMMENDATION = 10
top_recommendations = df_sorted.select("job_title", "company_name", "job_requirements", "category_scraped", "job_type", "min_salary", "max_salary", "min_experience_level", "education_req", "similarity_score").limit(TOP_RECOMMENDATION)
top_recommendations.show(truncate=False)

+-------------------------------+-----------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------+---------------------+---------+----------+----------+--------------------+-------------+----------------+
|job_title                      |company_name                       |job_requirements                                                                                                                            |category_scraped     |job_type |min_salary|max_salary|min_experience_level|education_req|similarity_score|
+-------------------------------+-----------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------+---------------------+---------+----------+----------+--------------------+-------------+----------------+
|Sales Admin Staff              |PT Tri Harmonis 

In [19]:
# stop spark session
spark.stop()